<a href="https://colab.research.google.com/github/ilchukmark/Analytics/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/ab_test_results.csv'
df = pd.read_csv(file_path)

print(df.head())
print(df['event_name'].unique())

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

metrics_definitions = {
    'add_payment_info': ('add_payment_info', 'session'),
    'add_shipping_info': ('add_shipping_info', 'session'),
    'begin_checkout': ('begin_checkout', 'session'),
    'new_accounts': ('new account', 'session')
}

def run_z_test(clicks_a, count_a, clicks_b, count_b):
    if count_a == 0 or count_b == 0:
        return 0.0, 0.0, 0.0, 0.0, 1.0, False

    p_a = clicks_a / count_a
    p_b = clicks_b / count_b

    metric_change = (abs(p_a - p_b) / p_a * 100) if p_a > 0 else 0.0

    p_combined = (clicks_a + clicks_b) / (count_a + count_b)
    if p_combined == 0 or p_combined == 1:
        return p_a, p_b, metric_change, 0.0, 1.0, False

    z_stat = (p_b - p_a) / np.sqrt(p_combined * (1 - p_combined) * (1 / count_a + 1 / count_b))
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    significant = p_value < 0.05

    return p_a, p_b, metric_change, abs(z_stat), p_value, significant

rows_list = []

for test_id in sorted(df['test'].unique(), reverse=True):
    test_df = df[df['test'] == test_id]

    pivot_seg = test_df.pivot_table(
        index='test_group',
        columns='event_name',
        values='value',
        aggfunc='sum'
    ).reset_index().fillna(0)

    if 'session' not in pivot_seg.columns or pivot_seg.empty:
        continue

    row_a = pivot_seg[pivot_seg['test_group'] == 1]
    row_b = pivot_seg[pivot_seg['test_group'] == 2]

    if row_a.empty or row_b.empty:
        continue

    count_a = row_a['session'].values[0]
    count_b = row_b['session'].values[0]

    for metric_label, (num_col, den_col) in metrics_definitions.items():
        clicks_a = row_a[num_col].values[0] if num_col in row_a.columns else 0
        clicks_b = row_b[num_col].values[0] if num_col in row_b.columns else 0

        cr_a, cr_b, metric_change, z_stat, p_value, significant = run_z_test(clicks_a, count_a, clicks_b, count_b)

        entry = {
            'test_number': test_id,
            'metric': metric_label,
            'numerator_event': num_col,
            'denominator_event': den_col,
            'numerator_count_a': clicks_a,
            'denominator_count_a': count_a,
            'conversion_rate_a': cr_a,
            'numerator_count_b': clicks_b,
            'denominator_count_b': count_b,
            'conversion_rate_b': cr_b,
            'metric_change_percent': metric_change,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': str(significant).upper()
        }

        rows_list.append(entry)

final_table = pd.DataFrame(rows_list)
final_table.to_csv('/content/drive/MyDrive/ab_test_processed_results.csv', index=False)

pd.set_option('display.max_columns', None)
display(final_table.head(10))

,test_number,metric,numerator_event,denominator_event,numerator_count_a,denominator_count_a,conversion_rate_a,numerator_count_b,denominator_count_b,conversion_rate_b,metric_change_percent,z_stat,p_value,significant
0,4,add_payment_info,add_payment_info,session,3731,105079,0.035507,3601,105141,0.034249,3.541234,1.571106,0.116158,FALSE
1,4,add_shipping_info,add_shipping_info,session,5128,105079,0.048801,4956,105141,0.047137,3.411125,1.785795,0.074132,FALSE
2,4,begin_checkout,begin_checkout,session,12555,105079,0.119482,12267,105141,0.116672,2.351523,1.995998,0.045934,TRUE
3,4,new_accounts,new account,session,8984,105079,0.085498,8687,105141,0.082622,3.362896,2.375457,0.017527,TRUE
4,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,FALSE
5,3,add_shipping_info,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,2.621211,1.413727,0.157442,FALSE
6,3,begin_checkout,begin_checkout,session,9532,70047,0.136080,9264,70439,0.131518,3.352445,2.511389,0.012026,TRUE
7,3,new_accounts,new account,session,5856,70047,0.083601,5822,70439,0.082653,1.133880,0.643489,0.519907,FALSE
8,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,FALSE
9,2,add_shipping_info,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,FALSE


Логіка скрипту полягає в послідовній фільтрації вхідного датасету за ідентифікатором тесту з подальшою трансформацією логів через зведену таблицю, де індексами виступають контрольна та тестова групи, а колонками - сумарні значення івентів. Для кожної визначеної метрики розраховуються індивідуальні конверсії груп як відношення цільового чисельника до загальної кількості сесій, а також обчислюється відсоткова величина відносної зміни між ними. Статистична значущість змін визначається за допомогою двовибіркового Z-критерію для часток, який на основі об'єднаної дисперсії генерує Z-статистику та відповідне P-значення, де поріг альфа фіксується на рівні п'яти відсотків.

Структура фінальної таблиці строго лінеаризована і містить ідентифікатори експерименту, текстові назви метрик та технічні назви сирих івентів чисельника і знаменника. Далі в межах одного рядка розташовані абсолютні обсяги конверсійних дій та сесій для обох груп, розраховані коефіцієнти конверсії, відсотковий ефект зміни, а також підсумкові метрики Z-тесту разом із булевим прапором значущості у верхньому регістрі.

[Переглянути візуалізацію можна тут](https://public.tableau.com/shared/ZD6Z3MN84?:display_count=n&:origin=viz_share_link)

[Посилання на файл](https://drive.google.com/file/d/1M6LXhkkNYm_1yFCbwqclFq483nTuG7pj/view?usp=sharing)